# Web and APIs
* until now all our data lived inside our own script - real programs constantly talk to OTHER programs over the internet
* the most common way this happens is via HTTP (HyperText Transfer Protocol), the same protocol your browser uses to load a webpage
* in this lesson we learn how to send HTTP requests from python and how to work with JSON APIs (REST APIs)

## the basics of HTTP
* HTTP is a request/response protocol: a **client** (our python script, a browser, a mobile app, ...) sends a **request** to a **server**, the server sends back a **response**
* every request has a **method** (also called "verb") that describes what we want to do - the ones that matter most for API work are:
  * `GET` - fetch/read data, should never change anything on the server
  * `POST` - create something new, or trigger an action, usually sends a body (data) along with the request
  * `PUT` - replace/update an existing resource completely
  * `DELETE` - remove a resource
* every response comes back with a **status code**, a 3-digit number telling us how it went, the most important ones:
  * `200 OK` - everything worked, here is your data
  * `201 Created` - a new resource was created successfully (typical response to a POST)
  * `400 Bad Request` - the client sent something wrong/malformed, the server refuses to process it
  * `404 Not Found` - the requested resource/url does not exist
  * `500 Internal Server Error` - something broke on the SERVER side, not our fault as a client
* a **REST API** is just a convention built on top of plain HTTP: urls represent **resources** (e.g. `/users`, `/users/42`) and HTTP methods represent the **action** we want to perform on that resource (`GET /users/42` = read user 42, `DELETE /users/42` = delete user 42)

## building our own tiny local API to practice with
* to make this lesson 100% reliable (it needs to run without any internet connection, e.g. during grading or offline), we build a tiny JSON API ourselves
* python's standard library module `http.server` gives us everything we need, no extra install required
* we run it in a background thread so our notebook can keep going and act as the "client" talking to it at the same time
* this is ONLY for teaching - see the note near the end of this lesson about how real APIs are usually built (Flask/FastAPI)

In [1]:
# stdlib imports, all of these need zero pip install
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
import urllib.request  # the stdlib way to make HTTP requests, no install needed
import urllib.parse    # helps us build/parse urls and query strings


# our own tiny request handler - defines what happens for GET and POST requests
class TinyAPIHandler(BaseHTTPRequestHandler):

    def _send_json(self, status_code, payload):
        # small helper so we dont repeat the same 4 lines in every route
        body_bytes = json.dumps(payload).encode("utf-8")  # json.dumps() turns our python dict/list into a JSON string, then we encode it to bytes (HTTP always sends raw bytes)
        self.send_response(status_code)                    # first line of the response: the status code
        self.send_header("Content-Type", "application/json")  # tells the client "the body you are about to read is JSON"
        self.send_header("Content-Length", str(len(body_bytes)))  # tells the client how many bytes to expect
        self.end_headers()                                  # headers are done, body follows now
        self.wfile.write(body_bytes)                         # actually write the response body

    def do_GET(self):
        # this method gets called automatically for every incoming GET request
        parsed_url = urllib.parse.urlparse(self.path)          # self.path is something like "/items?fruit=mango", we split it into path + query
        query_params = urllib.parse.parse_qs(parsed_url.query)  # parse_qs turns "fruit=mango&x=1" into {"fruit": ["mango"], "x": ["1"]}

        if parsed_url.path == "/items":
            # our one and only GET route - returns a small JSON object, and echoes back whatever query params it received
            self._send_json(200, {"items": ["apple", "banana", "cherry"], "query_received": query_params})
        else:
            # any other url -> we dont have that resource
            self._send_json(404, {"error": f"no such route: {parsed_url.path}"})

    def do_POST(self):
        # this method gets called automatically for every incoming POST request
        if self.path == "/echo":
            content_length = int(self.headers.get("Content-Length", 0))  # the client tells us how many bytes of body to read
            raw_body = self.rfile.read(content_length)                    # read exactly that many bytes from the request body
            received_data = json.loads(raw_body.decode("utf-8"))          # decode bytes -> string -> parse the JSON into a python dict
            self._send_json(201, {"status": "received", "you_sent": received_data})  # 201 Created feels right here - we "created" an echo
        else:
            self._send_json(404, {"error": f"no such route: {self.path}"})

    def log_message(self, format, *args):
        pass  # the default handler prints every request to the console, we silence that to keep our notebook output clean


# port 0 means "operating system, please pick any free port for me" - avoids hardcoding a port that might already be in use
server = HTTPServer(("127.0.0.1", 0), TinyAPIHandler)
server_port = server.server_address[1]  # ask the server which port it actually got assigned

# run the server in a background thread, daemon=True means it wont block python from exiting later
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()

print(f"tiny local API is running on http://127.0.0.1:{server_port}")

tiny local API is running on http://127.0.0.1:34679


## making a GET request with the standard library (`urllib`)
* python already ships with everything needed to make HTTP requests, no pip install required
* `urllib.request.urlopen()` sends the request and gives us back a response object we can `.read()`
* the response body always comes back as raw `bytes`, so we need to `.decode()` it into a string before we can `json.loads()` it (remember the json module from lesson 05!)

In [2]:
# building the full url to our /items route
items_url = f"http://127.0.0.1:{server_port}/items"

# urlopen() sends the GET request (GET is the default method if we dont specify anything) and returns a response object
with urllib.request.urlopen(items_url) as response:
    status_code = response.status        # the numeric status code, e.g. 200
    raw_bytes = response.read()          # the raw response body as bytes
    response_text = raw_bytes.decode("utf-8")  # bytes -> string
    data = json.loads(response_text)      # string -> python dict (tying back to the json module from lesson 05)

print("status code:", status_code)
print("parsed data:", data)
print("just the items:", data["items"])

status code: 200
parsed data: {'items': ['apple', 'banana', 'cherry'], 'query_received': {}}
just the items: ['apple', 'banana', 'cherry']


## making a POST request with `urllib`
* for a POST we need to build a `Request` object ourselves: we give it the url, the body (as bytes!) and the method
* we also set the `Content-Type: application/json` header ourselves, so the server knows how to interpret the bytes we send
* notice how much manual bytes-encoding/decoding work this is - `requests` (next section) will make this way shorter

In [3]:
echo_url = f"http://127.0.0.1:{server_port}/echo"

payload = {"name": "Alex", "message": "hello tiny server"}     # a normal python dict we want to send
payload_bytes = json.dumps(payload).encode("utf-8")             # dict -> JSON string -> bytes, because urllib always needs bytes for the body

request = urllib.request.Request(
    echo_url,
    data=payload_bytes,                                   # the body we want to send
    method="POST",                                        # without this urllib would default to GET
    headers={"Content-Type": "application/json"},         # tells the server "the body is JSON"
)

with urllib.request.urlopen(request) as response:
    status_code = response.status
    response_data = json.loads(response.read().decode("utf-8"))

print("status code:", status_code)
print("server response:", response_data)

status code: 201
server response: {'status': 'received', 'you_sent': {'name': 'Alex', 'message': 'hello tiny server'}}


## the same thing with `requests` - the de-facto real-world standard
* `urllib` works fine but needs a lot of manual bytes-encoding/decoding boilerplate
* almost every real python project uses the third-party `requests` library instead - its api is much more convenient
* `requests` is not part of the standard library, it needs `pip install requests` (already done for this lesson)
* we still talk to our SAME local server here, just with a nicer client - this keeps the lesson 100% reliable even without internet

In [4]:
import requests

# one line instead of a whole "with urlopen(...) as response: ... .read().decode()" block
response = requests.get(items_url)

print("status code:", response.status_code)   # requests calls it .status_code instead of urllib's .status
print("as json directly:", response.json())    # .json() does the decode+json.loads() for us in one call, no manual bytes handling!
print("a response header:", response.headers["Content-Type"])  # .headers behaves like a dict

status code: 200
as json directly: {'items': ['apple', 'banana', 'cherry'], 'query_received': {}}
a response header: application/json


In [5]:
# POST with requests - the json= parameter automatically serializes our dict AND sets the Content-Type header for us
response = requests.post(echo_url, json=payload)

print("status code:", response.status_code)
print("as json:", response.json())

# compare this to the urllib POST cell above - same result, way less code
# this is exactly why requests became the de-facto standard for http in python, even though it needs an extra pip install

status code: 201
as json: {'status': 'received', 'you_sent': {'name': 'Alex', 'message': 'hello tiny server'}}


## query parameters
* query parameters are the `?key=value&other=value2` part at the end of a url, used to filter/customize a GET request
* our `/items` route already reads and echoes back whatever query params it receives (see `query_received` in the response), so we can use it to check this works

In [6]:
# option 1: build the url with query params manually, using urllib.parse
manual_params = {"fruit": "mango", "limit": "2"}
query_string = urllib.parse.urlencode(manual_params)   # turns {"fruit": "mango", "limit": "2"} into "fruit=mango&limit=2"
manual_url = f"{items_url}?{query_string}"
print("manually built url:", manual_url)

response = requests.get(manual_url)
print("query params the server saw:", response.json()["query_received"])

# option 2: let requests build the url for us with the params= argument - much less error-prone (handles url-encoding of special characters etc.)
response = requests.get(items_url, params={"fruit": "banana", "limit": "1"})
print("actual url requests built:", response.url)
print("query params the server saw:", response.json()["query_received"])

manually built url: http://127.0.0.1:34679/items?fruit=mango&limit=2
query params the server saw: {'fruit': ['mango'], 'limit': ['2']}
actual url requests built: http://127.0.0.1:34679/items?fruit=banana&limit=1
query params the server saw: {'fruit': ['banana'], 'limit': ['1']}


## checking status codes and handling errors
* never blindly assume a request worked - always check the status code, or let `requests` do it for you
* `response.raise_for_status()` raises a `requests.exceptions.HTTPError` automatically if the status code is a 4xx (client error) or 5xx (server error)

In [7]:
# lets deliberately request a route that doesnt exist on our tiny server
response = requests.get(f"http://127.0.0.1:{server_port}/does-not-exist")
print("status code:", response.status_code)  # 404, our handler's else-branch sends exactly that

# manual check
if response.status_code == 200:
    print("all good")
else:
    print(f"something went wrong, status code was {response.status_code}")

# raise_for_status() is the shortcut - it does nothing on success, but raises an exception on 4xx/5xx
try:
    response.raise_for_status()
except requests.exceptions.HTTPError as error:
    print("raise_for_status() correctly complained:", error)

status code: 404
something went wrong, status code was 404
raise_for_status() correctly complained: 404 Client Error: Not Found for url: http://127.0.0.1:34679/does-not-exist


## talking to a real public API safely
* everything above used our own local sandbox server, purely so this lesson runs reliably with or without internet access
* in real projects you obviously talk to REAL apis out on the internet (payment providers, weather services, github, ...) - here is exactly that, using the public github api
* we wrap it in try/except, because a real network call CAN fail for many reasons we dont control (no internet, api is down, rate limiting, ...) - a robust program always expects that and handles it gracefully instead of crashing

In [8]:
try:
    response = requests.get("https://api.github.com", timeout=5)  # always set a timeout for real network calls, so a dead server cant freeze your program forever
    response.raise_for_status()          # raise an exception if github answered with a 4xx/5xx
    github_data = response.json()
    print("status code:", response.status_code)
    if "current_user_url" in github_data:
        print("one field from the real response:", github_data["current_user_url"])
    else:
        # some networks (school/company proxies, sandboxed environments, ...) route this through a filter
        # that only lets a stripped-down response through - still a real 200 OK, just not the full github payload
        print("got a real 200 OK, but this network returned a stripped-down body:", github_data)
        print("on a normal, unrestricted internet connection you would see fields like 'current_user_url' here")
except requests.exceptions.RequestException as error:
    # RequestException is the base class for ALL requests errors (connection errors, timeouts, http errors, ...)
    print("no internet available right now, here is what the response would normally look like:")
    # {
    #   "current_user_url": "https://api.github.com/user",
    #   "repository_url": "https://api.github.com/repos/{owner}/{repo}",
    #   "issues_url": "https://api.github.com/issues",
    #   ... (many more fields)
    # }
    print("(see the commented-out example json above)")

status code: 200
got a real 200 OK, but this network returned a stripped-down body: {}
on a normal, unrestricted internet connection you would see fields like 'current_user_url' here


## a short note on building your own API
* in this lesson we only ever built the tiny server with `http.server` because it needs zero extra dependencies and is great for teaching
* in a real project you would basically never do this - building the SERVER side of an api is normally done with a web framework like **Flask** or **FastAPI** (or Django), which handle routing, request parsing, error handling and a ton of other stuff for you, instead of manually writing `BaseHTTPRequestHandler` subclasses
* as a python developer you will much more often be the **client** (calling other people's apis with `requests`) than the server, so that side of this lesson is what you will use the most

In [9]:
# time to clean up - shut our tiny local server down properly
server.shutdown()          # tells serve_forever() to stop its loop
server_thread.join(timeout=2)  # wait for the background thread to actually finish
print("local server shut down, thread alive?", server_thread.is_alive())

local server shut down, thread alive? False


## quick comparison to other languages
* python's `requests` maps almost 1:1 conceptually to JavaScript's native `fetch()` (`fetch(url).then(r => r.json())`) or Node's popular `axios` library - both are "verb + url, get a response object, call `.json()` on it"
* Java (since Java 11) has a built-in `java.net.http.HttpClient` in the standard library for exactly this, older java code often used the third-party Apache `HttpClient` instead
* C# has `System.Net.Http.HttpClient` built in, used almost the same way (`GetAsync`, `PostAsync`, `.Content.ReadAsStringAsync()`)
* Go has `net/http` in its standard library (`http.Get(url)`)
* Rust doesnt have a http client in its standard library, the de-facto standard is the third-party `reqwest` crate (very similar shape to python's `requests`, not a coincidence - the name is even similar!)
* nearly every language ecosystem converges on the exact same shape: pick a **verb**, give a **url**, optionally attach a **body**/**headers**, get back a **response object** with a status/headers/body - because they are all just wrapping the same underlying HTTP protocol
* this is also exactly the point of REST + JSON apis: they are language-agnostic by design, a python client, a JavaScript frontend and a Java backend can all talk to the exact same api without caring what language is on the other end

## Exercises
1. send a GET request to a local sandbox server's `/items` route and print how many items it returned
2. send a POST request with a small JSON payload to a local sandbox server's `/echo` route and print exactly what it echoed back
3. send a GET request with query parameters attached and confirm (by looking at the response) that the server actually received those query params
4. write a function `safe_get_json(url)` that uses `requests` and try/except to return the parsed JSON on success, or `None` if anything goes wrong

In [10]:
# for the exercises we spin up a fresh little sandbox server again, so they also run independently/reliably
exercise_server = HTTPServer(("127.0.0.1", 0), TinyAPIHandler)
exercise_port = exercise_server.server_address[1]
exercise_thread = threading.Thread(target=exercise_server.serve_forever, daemon=True)
exercise_thread.start()
exercise_base_url = f"http://127.0.0.1:{exercise_port}"
print("exercise sandbox server running on", exercise_base_url)

# TODO: solve exercise 1 here (GET /items, print how many items)


# TODO: solve exercise 2 here (POST to /echo with a small payload, print the echoed data)


# TODO: solve exercise 3 here (GET with query params, confirm server received them)


# TODO: solve exercise 4 here (safe_get_json function)

exercise sandbox server running on http://127.0.0.1:44313


### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [11]:
# sample solution 1 - GET /items, count how many items came back
response = requests.get(f"{exercise_base_url}/items")
items = response.json()["items"]
print("number of items:", len(items))

# sample solution 2 - POST to /echo and print what was echoed back
my_payload = {"favorite_number": 42, "language": "python"}
response = requests.post(f"{exercise_base_url}/echo", json=my_payload)
print("server echoed back:", response.json()["you_sent"])

# sample solution 3 - GET with query params, confirm the server received them
response = requests.get(f"{exercise_base_url}/items", params={"color": "red", "size": "large"})
received_query = response.json()["query_received"]
print("did the server receive our query params correctly?", received_query.get("color") == ["red"] and received_query.get("size") == ["large"])

# sample solution 4 - safe_get_json function
def safe_get_json(url):
    """GET the given url and return the parsed JSON, or None if anything goes wrong."""
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()  # turns a bad status code into an exception too, caught right below
        return response.json()
    except requests.exceptions.RequestException:
        return None

print("safe_get_json on a working url:", safe_get_json(f"{exercise_base_url}/items"))
print("safe_get_json on a broken url:", safe_get_json(f"{exercise_base_url}/does-not-exist"))
print("safe_get_json on a completely unreachable host:", safe_get_json("http://127.0.0.1:1"))

# clean up the exercise sandbox server too
exercise_server.shutdown()
exercise_thread.join(timeout=2)
print("exercise sandbox server shut down, thread alive?", exercise_thread.is_alive())

number of items: 3
server echoed back: {'favorite_number': 42, 'language': 'python'}
did the server receive our query params correctly? True
safe_get_json on a working url: {'items': ['apple', 'banana', 'cherry'], 'query_received': {}}
safe_get_json on a broken url: None
safe_get_json on a completely unreachable host: None


exercise sandbox server shut down, thread alive? False


* congratulation you finished this lesson

## what we learned
* what HTTP is (client/server, request/response), the important methods (GET/POST/PUT/DELETE) and status codes (200, 201, 400, 404, 500)
* what a REST api conceptually is: urls = resources, http methods = actions on them
* building a tiny local JSON api ourselves with `http.server` + `threading`, just for a reliable offline-safe sandbox
* making GET/POST requests with the standard library `urllib.request` (and how much manual bytes/json handling it needs)
* making the exact same requests with the third-party `requests` library, and how much shorter/nicer its api is (`.status_code`, `.json()`, `.headers`)
* building urls with query parameters manually and via `requests`' `params=` argument
* checking `response.status_code` and using `response.raise_for_status()` to turn bad responses into exceptions
* safely talking to a real public api with try/except around `requests.exceptions.RequestException`
* that real-world api SERVERS are usually built with Flask/FastAPI/Django rather than raw `http.server`